# Last Training Run — Analysis & Evaluation
Auto-detects the most recent `run_*` folder under `models/upper_ppo_3gnb/`.

In [ ]:
import os, sys, json, csv
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── locate repo root ──────────────────────────────────────────────────────────
REPO = Path("__file__").resolve().parent.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# ── find latest run ───────────────────────────────────────────────────────────
MODEL_BASE = REPO / "models" / "upper_ppo_3gnb"
runs = sorted(MODEL_BASE.glob("run_*"), key=lambda p: p.stat().st_mtime, reverse=True)
assert runs, f"No runs found under {MODEL_BASE}"
RUN = runs[0]
print(f"Latest run: {RUN.name}")

TRAIN_CSV  = RUN / "training_log.csv"
VAL_CSV    = RUN / "validation_log.csv"
CONFIG     = json.loads((RUN / "config.json").read_text())
BEST_MODEL = RUN / "upper_ppo_best.zip"
FINAL_MODEL= RUN / "upper_ppo_final.zip"

print(f"Timesteps : {CONFIG['total_timesteps']}")
print(f"Scenario  : {CONFIG['training_scenarios']}")
print(f"LR        : {CONFIG['learning_rate']}")
print(f"n_steps   : {CONFIG['ppo_n_steps']}")

In [ ]:
# ── load training CSV ─────────────────────────────────────────────────────────
with TRAIN_CSV.open(newline="", encoding="utf-8") as f:
    train_rows = list(csv.DictReader(f))

def s(field, rows=None):
    rows = rows or train_rows
    out = []
    for r in rows:
        try:    out.append(float(r.get(field, 0) or 0))
        except: out.append(0.0)
    return np.array(out)

def rolling(arr, w=500):
    w = min(w, len(arr))
    cs = np.cumsum(np.insert(arr, 0, 0.0))
    out = np.empty_like(arr)
    for i in range(len(arr)):
        st = max(0, i + 1 - w)
        out[i] = (cs[i+1] - cs[st]) / (i + 1 - st)
    return out

steps    = s("step")
reward   = s("reward")
imb_end  = s("load_imbalance_end")
imb_st   = s("load_imbalance_start")
handover = s("handover_count")
sla_cnt  = s("sla_count")
overload = s("overload_ratio")
cost_imp = s("global_cost_improvement")

r_load = s("reward_load_improvement")
r_sla  = s("reward_sla_improvement")
r_sat  = s("reward_saturation_improvement")
r_neu  = s("reward_neutral_bias_penalty")
r_wrg  = s("reward_wrong_bias_penalty")
r_lbl  = s("reward_load_balance_level_bonus")

SLICE_TYPES = ["eMBB", "URLLC", "mMTC"]
GNB_IDS     = [0, 1, 2]
NEIGHBORS   = {0:(1,2), 1:(0,2), 2:(0,1)}
bias_series = {}
for src in GNB_IDS:
    for tgt in NEIGHBORS[src]:
        for sl in SLICE_TYPES:
            key = f"bias_g{src}_to_g{tgt}_{sl}"
            bias_series[key] = s(key)

W = 500
print(f"Rows: {len(train_rows)}  |  Max step: {int(steps.max())}")

## 1 · Reward & Load-Balancing Overview

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

# Reward
ax = axes[0, 0]
ax.plot(steps, reward, color="#8ecae6", alpha=0.15, lw=0.6)
ax.plot(steps, rolling(reward, W), color="#023047", lw=2.0, label=f"rolling {W}")
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set(title="Step reward", ylabel="reward")
ax.legend(fontsize=8)

# Load imbalance
ax = axes[0, 1]
ax.plot(steps, rolling(imb_st, W), color="#e9c46a", lw=1.5, ls="--", label="start")
ax.plot(steps, rolling(imb_end, W), color="#d62828", lw=2.0, label="end")
ax.set(title="Load imbalance", ylabel="imbalance")
ax.legend(fontsize=8)

# Handovers
ax = axes[1, 0]
ax.plot(steps, rolling(handover, W), color="#6a4c93", lw=2.0, label="handovers/step")
ax.set(title="Handovers per step", xlabel="training step", ylabel="count")
ax.legend(fontsize=8)

# Overload ratio & SLA count
ax = axes[1, 1]
ax.plot(steps, rolling(overload, W), color="#e76f51", lw=2.0, label="overload ratio")
ax.plot(steps, rolling(sla_cnt, W) / max(sla_cnt.max(), 1), color="#2a9d8f", lw=2.0, label="SLA count (norm)")
ax.set(title="Overload & SLA", xlabel="training step", ylabel="fraction")
ax.legend(fontsize=8)

for ax in axes.flat:
    ax.grid(alpha=0.25)
fig.suptitle(f"Training overview — {RUN.name}", fontsize=13)
fig.tight_layout()
plt.show()

## 2 · Reward Component Breakdown

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharex=True)
components = [
    (r_load, "reward_load_improvement",         "#023047"),
    (r_sla,  "reward_sla_improvement",           "#2a9d8f"),
    (r_sat,  "reward_saturation_improvement",    "#e76f51"),
    (r_neu,  "reward_neutral_bias_penalty",       "#6a4c93"),
    (r_wrg,  "reward_wrong_bias_penalty",         "#d62828"),
    (r_lbl,  "reward_load_balance_level_bonus",   "#e9c46a"),
]
for ax, (arr, label, col) in zip(axes.flat, components):
    ax.plot(steps, arr, color=col, alpha=0.12, lw=0.5)
    ax.plot(steps, rolling(arr, W), color=col, lw=2.0)
    ax.axhline(0, color="k", lw=0.7, ls="--")
    ax.set(title=label.replace("reward_",""), ylabel="value")
    ax.grid(alpha=0.25)
for ax in axes[1]:
    ax.set_xlabel("training step")
fig.suptitle("Reward components", fontsize=13)
fig.tight_layout()
plt.show()

## 3 · Learned Directional Biases

In [ ]:
COLORS = {"eMBB": "#023047", "URLLC": "#e76f51", "mMTC": "#2a9d8f"}
pairs = [(src, tgt) for src in GNB_IDS for tgt in NEIGHBORS[src]]
n_pairs = len(pairs)

fig, axes = plt.subplots(2, n_pairs // 2, figsize=(16, 7), sharex=True, sharey=True)
for ax, (src, tgt) in zip(axes.flat, pairs):
    for sl in SLICE_TYPES:
        key = f"bias_g{src}_to_g{tgt}_{sl}"
        ax.plot(steps, rolling(bias_series[key], W), color=COLORS[sl], lw=2.0, label=sl)
    ax.axhline(0, color="k", lw=0.8, ls="--")
    ax.set(title=f"g{src} → g{tgt}")
    ax.grid(alpha=0.25)
    ax.set_ylim(-1.05, 1.05)
axes[0, 0].legend(fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("training step")
for ax in axes[:, 0]:
    ax.set_ylabel("bias")
fig.suptitle("Directional biases per gNB pair & slice", fontsize=13)
fig.tight_layout()
plt.show()

## 4 · Per-gNB Load Evolution (end of step)

In [ ]:
GNB_COLORS = ["#023047", "#e76f51", "#2a9d8f"]

fig, axes = plt.subplots(1, len(SLICE_TYPES), figsize=(16, 4), sharex=True, sharey=True)
for ax, sl in zip(axes, SLICE_TYPES):
    for g, col in zip(GNB_IDS, GNB_COLORS):
        load = s(f"load_end_g{g}_{sl}")
        ax.plot(steps, rolling(load, W), color=col, lw=2.0, label=f"gNB-{g}")
    ax.axhline(0.85, color="red", lw=0.8, ls="--", label="overload thresh")
    ax.set(title=f"{sl} load (end)", xlabel="training step")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.25)
axes[0].set_ylabel("load")
axes[0].legend(fontsize=8)
fig.suptitle("Per-gNB per-slice load at end of upper window", fontsize=13)
fig.tight_layout()
plt.show()

## 5 · Evaluate Best & Final Models

In [ ]:
import argparse
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor as SB3Monitor

sys.path.insert(0, str(REPO))
from global_ppo_3gnb_env import GlobalPPO3GNBEnv, SLICE_TYPES as ENV_SLICE_TYPES
from upper_agent_training_scenarios import CENTER_GAP_GNB_CONFIGS

N_EVAL = 20   # episodes per model

def build_env(cfg):
    env = GlobalPPO3GNBEnv(
        seed=cfg["seed"],
        n_gnbs=cfg["n_gnbs"],
        slice_types=cfg["slice_types"],
        include_ue_counts=cfg.get("include_ue_counts", True),
        include_service_metrics=cfg.get("include_service_metrics", False),
        use_sumo_mobility=cfg.get("use_sumo_mobility", False),
        radio_substeps=cfg["radio_substeps"],
        radio_tick_seconds=cfg["radio_tick_seconds"],
        gnb_configs=CENTER_GAP_GNB_CONFIGS[cfg["center_gap_topology"]],
        local_steps_per_global=cfg["local_steps_per_global"],
        global_steps_per_episode=cfg["global_steps_per_episode"],
        scenario_mode=cfg["scenario_mode"],
        snapshot_scenario=cfg["snapshot_scenario"],
        terminal_reward_only=cfg["terminal_reward_only"],
        use_progress_reward=cfg["use_progress_reward"],
        max_handovers_per_local_step=cfg["max_handovers_per_local_step"],
        max_handovers_per_ue_episode=cfg["max_handovers_per_ue_episode"],
        max_handovers_per_episode=cfg["max_handovers_per_episode"],
        handover_pingpong_guard_s=cfg["handover_pingpong_guard_s"],
        action_direction_reward_weight=cfg["action_direction_reward_weight"],
        snapshot_block_episodes=cfg["snapshot_block_episodes"],
        light_load_ues=cfg["light_load_ues"],
        medium_load_ues=cfg["medium_load_ues"],
        high_load_ues=cfg["high_load_ues"],
        directional_global_action=True,
        global_reward_mu=cfg.get("load_balance_reward_weight", 2.0),
        global_reward_zeta=cfg.get("saturation_reward_weight", 1.0),
        global_reward_beta=cfg.get("sla_reward_weight", 1.0),
        global_action_kappa=cfg.get("bias_smoothing_weight", 0.01),
        global_action_lambda=cfg.get("negative_bias_penalty_weight", 0.01),
        sla_deadband=cfg["sla_deadband"],
        upper_window_seconds=cfg["upper_window_seconds"],
        training_scenarios=cfg["training_scenarios"],
        scenario_selection=cfg.get("scenario_selection", "cycle"),
        fixed_stage_episodes=cfg.get("fixed_stage_episodes", 500),
        slow_stage_episodes=cfg.get("slow_stage_episodes", 1000),
        global_neutral_bias_weight=cfg.get("global_neutral_bias_weight", 0.1),
        neutral_bias_eps=cfg.get("neutral_bias_eps", 0.05),
        wrong_bias_penalty_weight=cfg.get("wrong_bias_penalty_weight", 0.05),
        global_bad_direction_eta=cfg.get("global_bad_direction_eta", 0.025),
        global_unsafe_target_rho=cfg.get("global_unsafe_target_rho", 0.05),
        sla_severity_level_weight=cfg.get("sla_severity_level_weight", 0.1),
        load_balance_level_weight=cfg.get("load_balance_level_weight", 1.0),
        a3_handover_cooldown_s=cfg.get("a3_handover_cooldown_s", 2.0),
        a3_min_residence_s=cfg.get("a3_min_residence_s", 2.0),
        a3_history_window_s=cfg.get("a3_history_window_s", 20.0),
        a3_pingpong_threshold_s=cfg.get("a3_pingpong_threshold_s", 5.0),
        safe_admission_enabled=cfg.get("safe_admission", False),
        warmup_steps=cfg.get("warmup_steps", 0),
    )
    return SB3Monitor(env)


def run_eval(model, env, n_episodes):
    records = []
    for ep in range(n_episodes):
        obs, info = env.reset()
        done = False
        ep_ret = 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, rew, term, trunc, info = env.step(action)
            done = bool(term or trunc)
            ep_ret += float(rew)
            records.append({
                "episode": ep,
                "reward": float(rew),
                "load_imbalance_start": float(info.get("load_imbalance_start", 0)),
                "load_imbalance_end":   float(info.get("load_imbalance_end", 0)),
                "handover_count":       int(info.get("handover_count", 0)),
                "sla_count":            float(info.get("sla_count", 0)),
                "overload_ratio":       float(info.get("overload_ratio", 0)),
                "global_cost_improvement": float(info.get("global_cost_improvement", 0)),
                "scenario_name":        str(info.get("scenario_name", "")),
                "directional_bias_tensor": np.array(info.get("directional_bias_tensor", []), dtype=float),
                "load_matrix_end":      np.array(info.get("load_matrix_end", []), dtype=float),
            })
    return records


print("Loading models…")
best_model  = PPO.load(BEST_MODEL)
final_model = PPO.load(FINAL_MODEL)
print("Done.")

In [ ]:
print(f"Evaluating best model ({N_EVAL} episodes)…")
env_best = build_env(CONFIG)
best_records = run_eval(best_model, env_best, N_EVAL)
env_best.close()

print(f"Evaluating final model ({N_EVAL} episodes)…")
env_final = build_env(CONFIG)
final_records = run_eval(final_model, env_final, N_EVAL)
env_final.close()

print("Evaluation done.")

In [ ]:
def agg(records, key):
    return np.array([r[key] for r in records])

def summarise(records, label):
    rewards = agg(records, "reward")
    imb_red = agg(records, "load_imbalance_start") - agg(records, "load_imbalance_end")
    hos     = agg(records, "handover_count")
    sla     = agg(records, "sla_count")
    olr     = agg(records, "overload_ratio")
    print(f"\n── {label} ──")
    print(f"  reward          : {rewards.mean():.4f} ± {rewards.std():.4f}")
    print(f"  imbalance Δ     : {imb_red.mean():.4f} ± {imb_red.std():.4f}")
    print(f"  handovers/step  : {hos.mean():.2f}")
    print(f"  SLA count       : {sla.mean():.2f}")
    print(f"  overload ratio  : {olr.mean():.4f}")

summarise(best_records,  "Best model")
summarise(final_records, "Final model")

## 6 · Evaluation Plots

In [ ]:
labels = ["Best", "Final"]
all_recs = [best_records, final_records]
pal = ["#023047", "#e76f51"]

metrics = [
    ("reward",             "Reward"),
    ("load_imbalance_end", "Imbalance (end)"),
    ("handover_count",     "Handovers/step"),
    ("overload_ratio",     "Overload ratio"),
]

fig, axes = plt.subplots(1, len(metrics), figsize=(16, 4))
for ax, (key, title) in zip(axes, metrics):
    for recs, label, col in zip(all_recs, labels, pal):
        vals = agg(recs, key)
        eps  = np.arange(len(vals))
        ax.plot(eps, vals, color=col, lw=1.8, label=label)
    ax.set(title=title, xlabel="episode")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
fig.suptitle("Evaluation: best vs. final model", fontsize=13)
fig.tight_layout()
plt.show()

## 7 · Action Heatmap (best model)

In [ ]:
# Average directional bias tensor across eval episodes (best model)
bias_tensors = [r["directional_bias_tensor"] for r in best_records
                if r["directional_bias_tensor"].size > 0]

if bias_tensors:
    mean_bias = np.mean([t.reshape(3, 2, 3) for t in bias_tensors], axis=0)  # (n_gnbs, max_neighbors, n_slices)

    n_slices = len(SLICE_TYPES)
    fig, axes = plt.subplots(1, n_slices, figsize=(12, 3))
    for ax, sl_idx, sl in zip(axes, range(n_slices), SLICE_TYPES):
        data = mean_bias[:, :, sl_idx]          # shape (3, 2)
        im = ax.imshow(data, vmin=-1, vmax=1, cmap="RdBu", aspect="auto")
        ax.set_title(f"{sl}")
        ax.set_yticks([0,1,2]); ax.set_yticklabels([f"gNB-{g}" for g in GNB_IDS])
        ax.set_xticks([0,1]);   ax.set_xticklabels(["neighbor-0", "neighbor-1"])
        for r in range(3):
            for c in range(2):
                ax.text(c, r, f"{data[r,c]:.2f}", ha="center", va="center", fontsize=9,
                        color="white" if abs(data[r,c]) > 0.5 else "black")
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle("Mean directional bias — best model eval", fontsize=12)
    fig.tight_layout()
    plt.show()
else:
    print("No directional bias tensors found in eval records.")

## 8 · Validation CSV (from training run)

In [ ]:
if VAL_CSV.exists():
    with VAL_CSV.open(newline="", encoding="utf-8") as f:
        val_rows = list(csv.DictReader(f))
    print(f"Validation rows: {len(val_rows)}")

    val_reward  = np.array([float(r.get("reward", 0) or 0) for r in val_rows])
    val_imb_end = np.array([float(r.get("load_imbalance_end", 0) or 0) for r in val_rows])
    val_hos     = np.array([float(r.get("handover_count", 0) or 0) for r in val_rows])
    val_eps     = np.array([int(r.get("episode", 0) or 0) for r in val_rows])
    val_steps_v = np.array([int(r.get("step", 0) or 0) for r in val_rows])

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, arr, title, col in zip(
        axes,
        [val_reward, val_imb_end, val_hos],
        ["Reward", "Imbalance end", "Handovers"],
        ["#023047", "#d62828", "#6a4c93"]
    ):
        ax.plot(val_steps_v, arr, color=col, lw=1.5, marker=".", ms=3)
        ax.axhline(arr.mean(), color=col, ls="--", lw=1.0, label=f"mean={arr.mean():.3f}")
        ax.set(title=title, xlabel="step")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.25)
    fig.suptitle("Validation log (post-training)", fontsize=13)
    fig.tight_layout()
    plt.show()
else:
    print("No validation CSV found.")

## 9 · Run Summary

In [ ]:
val_cfg = CONFIG.get("validation", {})
print("=" * 50)
print(f"Run          : {RUN.name}")
print(f"Scenario     : {CONFIG['training_scenarios']}")
print(f"Timesteps    : {CONFIG['total_timesteps']}")
print(f"LR / n_steps : {CONFIG['learning_rate']} / {CONFIG['ppo_n_steps']}")
print("-" * 50)
print("Training (last-quarter step mean):")
n = len(reward)
lq = reward[3*n//4:]
print(f"  reward                : {lq.mean():.4f}")
print(f"  imbalance end (mean)  : {imb_end[3*n//4:].mean():.4f}")
print(f"  handovers (mean)      : {handover[3*n//4:].mean():.2f}")
print("-" * 50)
print("Post-training validation (config.json):")
for k, v in val_cfg.items():
    if k != "validation_csv":
        print(f"  {k:<40}: {v}")
print("=" * 50)

## 10 · Full Scenario Sweep — All Scenarios

In [ ]:
from upper_agent_training_scenarios import UPPER_TRAINING_SCENARIOS

N_SCENARIO_EVAL = 5   # episodes per scenario

def build_env_for_scenario(cfg, scenario_name):
    env = GlobalPPO3GNBEnv(
        seed=cfg["seed"],
        n_gnbs=cfg["n_gnbs"],
        slice_types=cfg["slice_types"],
        include_ue_counts=cfg.get("include_ue_counts", True),
        include_service_metrics=cfg.get("include_service_metrics", False),
        use_sumo_mobility=cfg.get("use_sumo_mobility", False),
        radio_substeps=cfg["radio_substeps"],
        radio_tick_seconds=cfg["radio_tick_seconds"],
        gnb_configs=CENTER_GAP_GNB_CONFIGS[cfg["center_gap_topology"]],
        local_steps_per_global=cfg["local_steps_per_global"],
        global_steps_per_episode=cfg["global_steps_per_episode"],
        scenario_mode="curriculum",
        snapshot_scenario=cfg["snapshot_scenario"],
        terminal_reward_only=cfg["terminal_reward_only"],
        use_progress_reward=cfg["use_progress_reward"],
        max_handovers_per_local_step=cfg["max_handovers_per_local_step"],
        max_handovers_per_ue_episode=cfg["max_handovers_per_ue_episode"],
        max_handovers_per_episode=cfg["max_handovers_per_episode"],
        handover_pingpong_guard_s=cfg["handover_pingpong_guard_s"],
        action_direction_reward_weight=cfg["action_direction_reward_weight"],
        snapshot_block_episodes=cfg["snapshot_block_episodes"],
        light_load_ues=cfg["light_load_ues"],
        medium_load_ues=cfg["medium_load_ues"],
        high_load_ues=cfg["high_load_ues"],
        directional_global_action=True,
        global_reward_mu=cfg.get("load_balance_reward_weight", 2.0),
        global_reward_zeta=cfg.get("saturation_reward_weight", 1.0),
        global_reward_beta=cfg.get("sla_reward_weight", 1.0),
        global_action_kappa=cfg.get("bias_smoothing_weight", 0.01),
        global_action_lambda=cfg.get("negative_bias_penalty_weight", 0.01),
        sla_deadband=cfg["sla_deadband"],
        upper_window_seconds=cfg["upper_window_seconds"],
        training_scenarios=scenario_name,
        scenario_selection="cycle",
        fixed_stage_episodes=cfg.get("fixed_stage_episodes", 500),
        slow_stage_episodes=cfg.get("slow_stage_episodes", 1000),
        global_neutral_bias_weight=cfg.get("global_neutral_bias_weight", 0.1),
        neutral_bias_eps=cfg.get("neutral_bias_eps", 0.05),
        wrong_bias_penalty_weight=cfg.get("wrong_bias_penalty_weight", 0.05),
        global_bad_direction_eta=cfg.get("global_bad_direction_eta", 0.025),
        global_unsafe_target_rho=cfg.get("global_unsafe_target_rho", 0.05),
        sla_severity_level_weight=cfg.get("sla_severity_level_weight", 0.1),
        load_balance_level_weight=cfg.get("load_balance_level_weight", 1.0),
        a3_handover_cooldown_s=cfg.get("a3_handover_cooldown_s", 2.0),
        a3_min_residence_s=cfg.get("a3_min_residence_s", 2.0),
        a3_history_window_s=cfg.get("a3_history_window_s", 20.0),
        a3_pingpong_threshold_s=cfg.get("a3_pingpong_threshold_s", 5.0),
        safe_admission_enabled=cfg.get("safe_admission", False),
        warmup_steps=cfg.get("warmup_steps", 0),
    )
    return SB3Monitor(env)


def run_eval_scenario(model, env, n_episodes):
    """
    Returns two lists:
      episode_records  – one dict per episode with episode-level before/after summary
      step_records     – one dict per upper-window step (for per-step trajectory plots)

    'before' = load state at episode reset, before the first agent action.
               Captured from the first step's load_matrix_start / load_imbalance_start.
    'after'  = load state at the end of the last upper window, after all agent actions.
               Captured from the last step's load_matrix_end / load_imbalance_end.

    For 1-window episodes both are identical to the single step's start/end.
    For 20-window episodes this correctly measures the net episode effect.
    """
    episode_records = []
    step_records    = []

    for ep in range(n_episodes):
        obs, _ = env.reset()
        done = False
        ep_ret = 0.0
        window_idx = 0

        # Episode-level accumulators
        initial_lm   = None   # load matrix BEFORE any agent action (first window start)
        initial_imb  = None
        final_lm     = None   # load matrix AFTER all agent actions (last window end)
        final_imb    = None
        total_ho     = 0
        total_sla    = 0.0
        step_imb_trajectory = []   # imbalance_end per window (for time-series plot)

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, rew, term, trunc, info = env.step(action)
            done = bool(term or trunc)
            ep_ret += float(rew)

            lm_st = np.array(info.get("load_matrix_start", []), dtype=float)
            lm_en = np.array(info.get("load_matrix_end",   []), dtype=float)
            lm_st = lm_st.reshape(3, 3) if lm_st.size == 9 else np.full((3,3), np.nan)
            lm_en = lm_en.reshape(3, 3) if lm_en.size == 9 else np.full((3,3), np.nan)
            imb_st = float(info.get("load_imbalance_start", np.nan))
            imb_en = float(info.get("load_imbalance_end",   np.nan))

            # Capture initial state from the very first window
            if window_idx == 0:
                initial_lm  = lm_st.copy()
                initial_imb = imb_st

            # Always update final state (overwrites each window; last wins)
            final_lm  = lm_en.copy()
            final_imb = imb_en

            total_ho  += int(info.get("handover_count", 0))
            total_sla += float(info.get("sla_count", 0))
            step_imb_trajectory.append(imb_en)

            step_records.append({
                "episode":    ep,
                "window":     window_idx,
                "reward":     float(rew),
                "imb_start":  imb_st,
                "imb_end":    imb_en,
                "handovers":  int(info.get("handover_count", 0)),
                "overload":   float(info.get("overload_ratio", 0)),
                "sla_count":  float(info.get("sla_count", 0)),
                "lm_start":   lm_st,
                "lm_end":     lm_en,
                "bias_tensor": np.array(info.get("directional_bias_tensor", []), dtype=float),
                "scenario":   str(info.get("scenario_name", "")),
            })
            window_idx += 1

        n_windows = window_idx
        episode_records.append({
            "episode":         ep,
            "n_windows":       n_windows,
            "ep_return":       ep_ret,
            "initial_lm":      initial_lm,   # load matrix BEFORE episode (first window start)
            "final_lm":        final_lm,     # load matrix AFTER episode (last window end)
            "initial_imb":     initial_imb,  # network imbalance before any action
            "final_imb":       final_imb,    # network imbalance after all actions
            "total_handovers": total_ho,
            "mean_sla":        total_sla / max(n_windows, 1),
            "imb_trajectory":  np.array(step_imb_trajectory),
        })

    return episode_records, step_records


ALL_SCENARIO_NAMES = [sc.name for sc in UPPER_TRAINING_SCENARIOS]
scenario_ep   = {}   # name → episode_records
scenario_step = {}   # name → step_records

print(f"Evaluating best model on {len(ALL_SCENARIO_NAMES)} scenarios × {N_SCENARIO_EVAL} episodes…")
for sc_name in ALL_SCENARIO_NAMES:
    env_sc = build_env_for_scenario(CONFIG, sc_name)
    ep_recs, st_recs = run_eval_scenario(best_model, env_sc, N_SCENARIO_EVAL)
    env_sc.close()
    scenario_ep[sc_name]   = ep_recs
    scenario_step[sc_name] = st_recs

    n_win   = np.mean([r["n_windows"] for r in ep_recs])
    imb_bef = np.mean([r["initial_imb"] for r in ep_recs])
    imb_aft = np.mean([r["final_imb"]   for r in ep_recs])
    ret     = np.mean([r["ep_return"]   for r in ep_recs])
    print(f"  {sc_name:<45}  windows={n_win:.0f}  "
          f"imb_before={imb_bef:.3f}  imb_after={imb_aft:.3f}  return={ret:.3f}")

print("Done.")

In [ ]:
# ── Cross-scenario summary — episode-level before/after ───────────────────────
sc_names    = list(scenario_ep.keys())
short_names = [n.replace("_", "\n") for n in sc_names]
TRAINED_ON  = CONFIG["training_scenarios"].split(",")

def ep_mean(sc, key):
    return np.mean([r[key] for r in scenario_ep[sc]])

mean_returns   = np.array([ep_mean(n, "ep_return")       for n in sc_names])
mean_imb_init  = np.array([ep_mean(n, "initial_imb")     for n in sc_names])  # BEFORE any action
mean_imb_final = np.array([ep_mean(n, "final_imb")       for n in sc_names])  # AFTER all actions
mean_imb_reduc = mean_imb_init - mean_imb_final
mean_ho        = np.array([ep_mean(n, "total_handovers") for n in sc_names])
mean_sla       = np.array([ep_mean(n, "mean_sla")        for n in sc_names])
mean_n_win     = np.array([ep_mean(n, "n_windows")       for n in sc_names])

bar_colors  = ["#2a9d8f" if n in TRAINED_ON else "#8ecae6" for n in sc_names]
edge_colors = ["#023047" if n in TRAINED_ON else "none"    for n in sc_names]
x = np.arange(len(sc_names))

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

for ax, vals, title, ylabel, yl in [
    (axes[0,0], mean_returns,   "Mean episode return",                        "return",        None),
    (axes[0,1], mean_imb_reduc, "Imbalance reduction\n(episode start → end)", "Δ imbalance",   None),
    (axes[1,0], mean_ho,        "Total handovers per episode",                 "count",         None),
    (axes[1,1], mean_sla,       "Mean SLA violations / window",               "SLA count",     None),
]:
    bars = ax.bar(x, vals, color=bar_colors, edgecolor=edge_colors, linewidth=1.5)
    ax.axhline(0, color="k", lw=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(short_names, fontsize=7, ha="center")
    ax.set(title=title, ylabel=ylabel)
    if yl: ax.set_ylim(*yl)
    ax.grid(axis="y", alpha=0.3)
    for bar, v in zip(bars, vals):
        ypos = bar.get_height() + (0.005 if v >= 0 else -0.01)
        ax.text(bar.get_x() + bar.get_width()/2, ypos, f"{v:.2f}",
                ha="center", va="bottom", fontsize=7)

from matplotlib.patches import Patch
legend_els = [Patch(facecolor="#2a9d8f", edgecolor="#023047", label="trained-on scenario"),
              Patch(facecolor="#8ecae6", edgecolor="none",    label="unseen scenario")]
fig.legend(handles=legend_els, loc="lower center", ncol=2, fontsize=9, frameon=True)
fig.suptitle("Best model — all scenarios (episode-level: before first action → after last action)", fontsize=13)
fig.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

## 11 · Before / After — Per-Scenario Load State

In [ ]:
# ── Before/After: per-gNB per-slice load for each scenario ────────────────────
# "before" = load matrix at episode reset  (first window's load_matrix_start)
# "after"  = load matrix at episode end    (last  window's load_matrix_end)
# For multi-window episodes this captures the NET effect of ALL agent actions.

SL_COLORS = {"eMBB": "#023047", "URLLC": "#e76f51", "mMTC": "#2a9d8f"}
SL_LIST   = ["eMBB", "URLLC", "mMTC"]

n_sc   = len(sc_names)
n_cols = 2
n_rows = (n_sc + 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4.5))
axes_flat = list(axes.flat)

for ax, sc_name in zip(axes_flat, sc_names):
    ep_recs = scenario_ep[sc_name]

    # Average initial and final load matrices over all eval episodes
    lm_before = np.nanmean([r["initial_lm"] for r in ep_recs], axis=0)  # (3, 3)
    lm_after  = np.nanmean([r["final_lm"]   for r in ep_recs], axis=0)
    n_win_avg = np.mean([r["n_windows"] for r in ep_recs])

    n_gnbs   = 3
    group_w  = 0.8
    bar_w    = group_w / (2 * len(SL_LIST) + 1)
    x_base   = np.arange(n_gnbs)

    for sl_i, sl in enumerate(SL_LIST):
        offset_before = (sl_i - 1) * 2 * bar_w - bar_w * 0.5
        offset_after  = offset_before + bar_w
        ax.bar(x_base + offset_before, lm_before[:, sl_i],
               width=bar_w, color="#cccccc", edgecolor=SL_COLORS[sl], linewidth=1.2,
               label=f"{sl} before" if ax is axes_flat[0] else "")
        ax.bar(x_base + offset_after, lm_after[:, sl_i],
               width=bar_w, color=SL_COLORS[sl], alpha=0.85,
               label=f"{sl} after" if ax is axes_flat[0] else "")

    ax.axhline(0.85, color="red", lw=1.0, ls="--",
               label="overload" if ax is axes_flat[0] else "")
    ax.set_xticks(x_base)
    ax.set_xticklabels([f"gNB-{g}" for g in range(n_gnbs)], fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("load")
    marker = "★ " if sc_name in TRAINED_ON else ""
    ax.set_title(f"{marker}{sc_name}  [{n_win_avg:.0f} windows/ep]",
                 fontsize=9, fontweight="bold" if sc_name in TRAINED_ON else "normal")
    ax.grid(axis="y", alpha=0.25)

    imb_bef = np.mean([r["initial_imb"] for r in ep_recs])
    imb_aft = np.mean([r["final_imb"]   for r in ep_recs])
    ax.text(0.98, 0.97, f"imb: {imb_bef:.3f} → {imb_aft:.3f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

for ax in axes_flat[len(sc_names):]:
    ax.set_visible(False)

axes_flat[0].legend(fontsize=8, ncol=2, loc="upper left",
                    title="slice  gray=before  color=after")
fig.suptitle(
    "Before / After load per gNB & slice\n"
    "before = episode initial state · after = episode final state · ★ trained-on",
    fontsize=13,
)
fig.tight_layout()
plt.show()

In [ ]:
# ── Before/After: imbalance summary + per-window trajectory for multi-step scenarios ─

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Left: grouped bars — episode initial vs final imbalance
ax = axes[0]
bw = 0.35
x  = np.arange(len(sc_names))

bars_b = ax.bar(x - bw/2, mean_imb_init,  width=bw, color="#cccccc",
                edgecolor="#444", linewidth=1.0, label="before (episode start)")
bars_a = ax.bar(x + bw/2, mean_imb_final, width=bw,
                color=bar_colors, edgecolor=edge_colors, linewidth=1.2,
                label="after (episode end)")

for xi, (b, a) in enumerate(zip(mean_imb_init, mean_imb_final)):
    col = "#2a9d8f" if a < b else "#d62828"
    ax.annotate("", xy=(xi + bw/2, a + 0.004), xytext=(xi - bw/2, b + 0.004),
                arrowprops=dict(arrowstyle="-|>", color=col, lw=1.3))

ax.set_xticks(x)
ax.set_xticklabels(sc_names, rotation=28, ha="right", fontsize=8)
ax.set(ylabel="load imbalance",
       title="Network imbalance — episode start vs episode end\n"
             "(before = pre-first-action · after = post-last-action)")
ax.set_ylim(0, max(mean_imb_init.max(), mean_imb_final.max()) * 1.25 + 0.02)
ax.grid(axis="y", alpha=0.3)
from matplotlib.patches import Patch
ax.legend(handles=[bars_b, bars_a,
                   Patch(facecolor="#2a9d8f", edgecolor="#023047", label="trained-on (after)"),
                   Patch(facecolor="#8ecae6", edgecolor="none",    label="unseen (after)")],
          fontsize=8, loc="upper right")

# Right: imbalance trajectory over windows for multi-window scenarios
ax2 = axes[1]
multi_win_scenarios = [n for n in sc_names
                       if np.mean([r["n_windows"] for r in scenario_ep[n]]) > 1]
cmap = plt.cm.tab10
for ci, sc_name in enumerate(multi_win_scenarios):
    trajs = [r["imb_trajectory"] for r in scenario_ep[sc_name]]
    max_len = max(len(t) for t in trajs)
    padded = np.full((len(trajs), max_len), np.nan)
    for i, t in enumerate(trajs):
        padded[i, :len(t)] = t
    mean_traj = np.nanmean(padded, axis=0)
    std_traj  = np.nanstd(padded,  axis=0)
    wins = np.arange(1, len(mean_traj) + 1)
    col  = cmap(ci % 10)
    ax2.plot(wins, mean_traj, color=col, lw=2.0,
             label=sc_name.replace("_", " "),
             marker="o" if len(wins) <= 5 else None, ms=5)
    ax2.fill_between(wins, mean_traj - std_traj, mean_traj + std_traj,
                     color=col, alpha=0.12)

if multi_win_scenarios:
    ax2.set(xlabel="upper window index",
            ylabel="load imbalance (end of window)",
            title="Imbalance trajectory over episode windows\n(multi-window scenarios only)")
    ax2.legend(fontsize=7, loc="upper right")
    ax2.grid(alpha=0.25)
else:
    ax2.text(0.5, 0.5, "All scenarios are 1-window episodes",
             ha="center", va="center", transform=ax2.transAxes)
    ax2.set_visible(True)

fig.tight_layout()
plt.show()